In [15]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict

### TripUpdates.pb

In [16]:
feed = gtfs_realtime_pb2.FeedMessage()
fpath = "TripUpdates_A.pb"
with open(fpath, 'rb') as f:
    feed.ParseFromString(f.read())

print(feed.header.gtfs_realtime_version)
print(len(feed.entity))

2.0
494


Co to mamy ciekawego w tripUpdates?

In [17]:
sample = feed.entity[0]
print(sample)

id: "trip_20261109_3_130095198_13"
trip_update {
  trip {
    trip_id: "20261109_3_130095198_13"
    schedule_relationship: SCHEDULED
    route_id: "126"
    direction_id: 1
  }
  stop_time_update {
    stop_sequence: 10
    arrival {
      delay: 103
      time: 1794410803
    }
    departure {
      delay: 103
      time: 1794410803
    }
    stop_id: "7538"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 11
    arrival {
      delay: 103
      time: 1794410863
    }
    departure {
      delay: 103
      time: 1794410863
    }
    stop_id: "10872"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 12
    arrival {
      delay: 103
      time: 1794410923
    }
    departure {
      delay: 103
      time: 1794410923
    }
    stop_id: "7537"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 13
    arrival {
      delay: 103
      time: 1794410983
    }
    departure {
      delay: 103
      t

Trip updates zawiera tylko informacje o tym co ma się zdarzyć i przewiduje opóźnienia metodą "to co jest aktualnie" - na każdym przystanku takie samo zapowiadane opóźnienie.

Kiedy autobus przyjedzie na przystanek fakty "co było" po prostu znikają. Trzeba wymyslic jak mierzyć kiedy autobus faktycznie był na przystanku - albo po tym, w którym updacie przystanek zniknął, tzn dla którego pliku .pb już przystanku nie ma i wtedy moment stworzenia pliku (albo ostatniego pobrania danych danego przejazdu - do przeanalizowania które lepsze)

In [18]:
print(f"Czas stworzenia pliku .pb: {feed.header.timestamp}")
print(f"Czas ostatniego odczytu próbkowanej trasy: {sample.trip_update.timestamp}")

Czas stworzenia pliku .pb: 1794410722
Czas ostatniego odczytu próbkowanej trasy: 1794410712


Każde entity można łatwo wrzucić do zagnieżdżonego słownika:

In [19]:
sample_entity_packed = MessageToDict(sample)
print(sample_entity_packed)

{'id': 'trip_20261109_3_130095198_13', 'tripUpdate': {'trip': {'tripId': '20261109_3_130095198_13', 'scheduleRelationship': 'SCHEDULED', 'routeId': '126', 'directionId': 1}, 'stopTimeUpdate': [{'stopSequence': 10, 'arrival': {'delay': 103, 'time': '1794410803'}, 'departure': {'delay': 103, 'time': '1794410803'}, 'stopId': '7538', 'scheduleRelationship': 'SCHEDULED'}, {'stopSequence': 11, 'arrival': {'delay': 103, 'time': '1794410863'}, 'departure': {'delay': 103, 'time': '1794410863'}, 'stopId': '10872', 'scheduleRelationship': 'SCHEDULED'}, {'stopSequence': 12, 'arrival': {'delay': 103, 'time': '1794410923'}, 'departure': {'delay': 103, 'time': '1794410923'}, 'stopId': '7537', 'scheduleRelationship': 'SCHEDULED'}, {'stopSequence': 13, 'arrival': {'delay': 103, 'time': '1794410983'}, 'departure': {'delay': 103, 'time': '1794410983'}, 'stopId': '7532', 'scheduleRelationship': 'SCHEDULED'}, {'stopSequence': 14, 'arrival': {'delay': 103, 'time': '1794411043'}, 'departure': {'delay': 103, 

Mozna też odwoływac się do niej bezpośrednio po nazwach pól - słownik po drodze wydaje się zbędny:

In [20]:
# np. sprawdzenie ile przystanków zostało:
print(len(sample.trip_update.stop_time_update))

20


In [21]:
print(len(feed.entity))

494


In [22]:
def unpack_pb_file(filepath: str) -> gtfs_realtime_pb2.FeedMessage: 
    feed = gtfs_realtime_pb2.FeedMessage()
    with open(filepath, 'rb') as f:
        feed.ParseFromString(f.read())
    return feed

In [25]:
alerts = unpack_pb_file("ServiceAlerts_A.pb")
alerts_count = len(alerts.entity)
if alerts_count > 0:
    print(f"Alert 1 z {alerts_count}: {alerts.entity[0]}")
else:
    print("brak alertów")

brak alertów


Wygląda na to że service alerts są raczej puste i nie potrzebujemy się nimi interesować

### VehiclePositions:

In [24]:
positions = unpack_pb_file("VehiclePositions_A.pb")
print(positions.entity[0])

id: "vehicle_PA103"
vehicle {
  trip {
    trip_id: "20261109_3_130095181_14"
    schedule_relationship: SCHEDULED
    route_id: "126"
    direction_id: 0
  }
  position {
    latitude: 50.0635
    longitude: 19.8746796
    bearing: 30
    speed: 17.5
  }
  current_stop_sequence: 13
  current_status: INCOMING_AT
  timestamp: 1794420916
  stop_id: "17970"
  vehicle {
    id: "PA103"
    label: "103"
    license_plate: "PA103"
  }
  occupancy_status: NO_DATA_AVAILABLE
}



To jest chyba najciekawszy z tych plików. Jest nawet prędkość w m/s i informacje jaki następny przystanek autobus będzie odwiedzał.

Wydaje mi się że poza "delay" to wszystko w zasadzie jest w tym pliku

## Podsumowanie

Pliki .pb są całkiem elegancko zapakowane i po otwarciu łatwo się z nimi pracuje. Są zaprojektowane pod wydajność, więc to je lepiej zbierać i otwierać/rozbijać dopiero jak będą potrzebne. 

Pliki ServiceAlerts możemy albo kompletnie zignorowac albo sprawdzać przed zapisaniem ich czy wgl jest niepusty - może coś się uda złapać i wtedy gdybyśmy w późniejszej wersji aplikacji chcieli dodać jakiegoś popupa w stylu "Uwaga! Taki a taki alert" to będziemy wiedzieli jaka jest jego struktura a nie musieli ufać dokumentacji/Geminiemu na słowo. 

TripUpdates i VehiclePositions przydałoby się zbierać często, bo częstotliwość zbierania w zasadzie równa się dokładności z jaką jesteśmy w stanie okreslić kiedy autobus przyjechał na przystanek, co przy aplikacji mierzącej spóźnienia pojazdu jest dosyć kluczowe.